# Automated Nigerian Media Subtitling Pipeline
**Models:** `NCAIR1/Hausa-ASR` · `NCAIR1/Yoruba-ASR` · `NCAIR1/Igbo-ASR` · `NCAIR1/NigerianAccentedEnglish`

# > **The Models Used**: [Hausa](https://huggingface.co/NCAIR1/Hausa-ASR) · [Yoruba](https://huggingface.co/NCAIR1/Yoruba-ASR) · [Igbo](https://huggingface.co/NCAIR1/Igbo-ASR) · [English](https://huggingface.co/NCAIR1/NigerianAccentedEnglish)

> **Runtime:** Set to T4 GPU so it will be fast

## Step 1: Install Dependencies

In [ ]:
!pip install -q --upgrade transformers accelerate
!pip install -q librosa soundfile pysrt ipywidgets hf_transfer
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
print(' Done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 46.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 131.2 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 631.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00
 Done.


## Step 2: HuggingFace Login

In [ ]:
import os

# SECURITY: never hardcode tokens in a notebook. Store it in Colab's Secrets
# manager (key icon in the left sidebar) under the name HF_TOKEN, or you'll be
# prompted to paste it once below (it won't be saved to the notebook file).
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Enter your Hugging Face token (input hidden): ')

os.environ['HF_TOKEN'] = HF_TOKEN
OUTPUT_SRT = 'output_subtitles.srt'
print('Token set.')


Enter your Hugging Face token (input hidden): ··········
Token set.


## Step 3: Download ASR Models
Run once. All models will be downloaded

In [ ]:
import os

# Upgrade the storage client first — this exact error is a known flaky bug in
# older hf-xet versions, usually fixed by upgrading + clearing any corrupted
# partial download left behind by a previous failed attempt.
!pip install -q -U huggingface_hub hf_xet
!rm -rf ~/.cache/huggingface/hub

from huggingface_hub import snapshot_download, hf_hub_download

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'

models = [
    'NCAIR1/Hausa-ASR',
    'NCAIR1/Yoruba-ASR',
    'NCAIR1/Igbo-ASR',
    'NCAIR1/NigerianAccentedEnglish'
]

import time

def download_with_retry(fn, *args, retries=3, **kwargs):
    # Xet occasionally throws a transient "Unable to parse string as hex hash
    # value" error mid-download. Lower concurrency + a short retry loop
    # papers over it without needing a manual re-run every time.
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            print(f'   attempt {attempt}/{retries} failed: {e}')
            if attempt == retries:
                raise
            time.sleep(4)

for m in models:
    print(f'Downloading {m}...')
    download_with_retry(snapshot_download, m, max_workers=2)
    print(f' {m} Downloaded.')

print("Downloading N-ATLaS GGUF (Q4_K_M)...")

model_path = download_with_retry(
    hf_hub_download,
    repo_id="tosinamuda/N-ATLaS-GGUF",
    filename="N-ATLaS-GGUF-Q4_K_M.gguf",
    resume_download=True,
)

print(f"Downloaded to: {model_path}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 77.1 MB/s eta 0:00:00


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

 NCAIR1/Hausa-ASR Downloaded.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

 NCAIR1/NigerianAccentedEnglish Downloaded.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(


N-ATLaS-GGUF-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 4.92GB            

N-ATLaS-GGUF-Q4_K_M.gguf: downloading bytes:           |  0.00B            

Downloaded to: /root/.cache/huggingface/hub/models--tosinamuda--N-ATLaS-GGUF/snapshots/af9195261dd163807b31318b9b9d577de5ae3b7e/N-ATLaS-GGUF-Q4_K_M.gguf


## Step 4: Auto-Transcribe, Translate & Caption (All-in-One)

Replaces the old separate Steps 4, 5, and 5.5. One cell now handles: video input → automatic spoken-language detection (with a chance to confirm/correct it) → pick a translation target → transcribe → clean up → translate → burn captions into the video. Run it once and answer the two prompts that appear.

In [ ]:
#@markdown ### Step 4: Auto-Transcribe, Translate & Caption (All-in-One)
#@markdown Upload a video (or point to a path), and this single cell will:
#@markdown 1. Automatically detect the spoken language
#@markdown 2. Ask you to confirm it (or correct it) and pick a translation target
#@markdown 3. Transcribe → clean up → translate → burn captions into the video
#@markdown
#@markdown One final captioned `.mp4` comes out the other end — nothing else to run.

input_method = "Upload"  #@param ["Upload", "File Path"]
file_path = "/content/Nigerian accented English.mp4"  #@param {type:"string"}

#@markdown **Subtitle timing**
max_subtitle_duration = 5.0   #@param {type:"number"}
max_words_per_subtitle = 12   #@param {type:"integer"}
pause_threshold = 0.6         #@param {type:"number"}

#@markdown **Cleanup / translation batching**
use_cleanup     = True   #@param {type:"boolean"}
batch_size      = 20     #@param {type:"integer"}
max_retries     = 2      #@param {type:"integer"}
batch_size_t    = 20     #@param {type:"integer"}
max_retries_t   = 2      #@param {type:"integer"}

#@markdown **Burned-in caption size**
subtitle_font_percent = 4.2  #@param {type:"slider", min:1.5, max:8, step:0.1}

# Imports
import subprocess, gc, torch, pysrt, os, re, json as _json_probe
import numpy as np
import librosa
from transformers import pipeline as hf_pipeline

# PART A — Video source

if input_method == 'Upload':
    from google.colab import files
    print('Select your video file...')
    uploaded = files.upload()
    VIDEO_PATH = list(uploaded.keys())[0]
else:
    assert os.path.exists(file_path), f'File not found: {file_path}'
    VIDEO_PATH = file_path

base_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
print(f' Video ready: {VIDEO_PATH}')
print(f' Size: {os.path.getsize(VIDEO_PATH) / (1024 * 1024):.1f} MB')

# PART B — Auto-detect spoken language, then confirm + pick translation target

NCAIR_MODELS = {
    'Hausa':            'NCAIR1/Hausa-ASR',
    'Yoruba':           'NCAIR1/Yoruba-ASR',
    'Igbo':             'NCAIR1/Igbo-ASR',
    'Nigerian English': 'NCAIR1/NigerianAccentedEnglish',
}
LANG_CODES = {'Hausa': 'ha', 'Yoruba': 'yo', 'Igbo': 'ig', 'Nigerian English': 'en'}
SUPPORTED_LANGS = ['Hausa', 'Yoruba', 'Igbo', 'Nigerian English']

print('\n[Language ID] Extracting a short audio sample for detection...')
LID_WAV = f'{base_name}_lid_sample.wav'
subprocess.run(
    ['ffmpeg', '-y', '-i', VIDEO_PATH, '-t', '30', '-ar', '16000', '-ac', '1',
     LID_WAV, '-loglevel', 'error'],
    check=True,
)
lid_audio, lid_sr = librosa.load(LID_WAV, sr=16000, mono=True)
os.remove(LID_WAV)

# Whisper's trained language set includes Hausa and Yoruba, but NOT Igbo — so Igbo
# audio can never be correctly auto-detected here. This is a real limitation, not a
# bug: if you know Igbo is being spoken, just correct it at the prompt below.
WHISPER_LANG_MAP = {'yo': 'Yoruba', 'ha': 'Hausa', 'en': 'Nigerian English'}

detected_language = None
try:
    print('[Language ID] Loading a small multilingual model just for detection...')
    from transformers import WhisperProcessor, WhisperForConditionalGeneration

    lid_processor = WhisperProcessor.from_pretrained('openai/whisper-base')
    lid_model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-base')
    if torch.cuda.is_available():
        lid_model = lid_model.to('cuda')

    lid_inputs = lid_processor(lid_audio, sampling_rate=16000, return_tensors='pt')
    lid_features = lid_inputs.input_features.to(lid_model.device)
    lang_token_ids = lid_model.detect_language(lid_features)
    detected_token = lid_processor.batch_decode(lang_token_ids)[0]
    whisper_code = detected_token.strip('<|>')
    detected_language = WHISPER_LANG_MAP.get(whisper_code)

    print(f'[Language ID] Whisper detected code "{whisper_code}"' +
          (f' -> {detected_language}' if detected_language
           else ' -> not one of our 4 supported languages (could be a misdetected '
                'Igbo clip, or genuine noise/silence)'))

    del lid_model, lid_processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as e:
    print(f'[Language ID] Auto-detection failed ({e}) — pick manually below.')

print('\n' + '=' * 55)
if detected_language:
    print(f'  Detected spoken language: {detected_language}')
else:
    print('  Could not confidently auto-detect the language.')
print('=' * 55)
print("Note: Igbo can't be auto-detected (see above) — double check if it's spoken.")

prompt_default = detected_language or 'Nigerian English'
user_input = input(
    f'\nPress Enter to accept "{prompt_default}", or type the correct language '
    f'({", ".join(SUPPORTED_LANGS)}): '
).strip()

if user_input:
    matches = [l for l in SUPPORTED_LANGS if l.lower() == user_input.lower()]
    language = matches[0] if matches else prompt_default
    if not matches:
        print(f'  "{user_input}" not recognized — using "{prompt_default}" instead.')
else:
    language = prompt_default
print(f'✅ Source language: {language}')

TARGET_OPTIONS = ['English', 'Hausa', 'Yoruba', 'Igbo']
target_input = input(
    f'\nWhich language should the translation be in? ({", ".join(TARGET_OPTIONS)}): '
).strip()
matches = [l for l in TARGET_OPTIONS if l.lower() == target_input.lower()]
if matches:
    target_language = matches[0]
else:
    target_language = 'English' if language != 'Nigerian English' else 'Hausa'
    print(f'  Not recognized (or left blank) — defaulting to "{target_language}".')
use_translation = True
print(f' Target language: {target_language}')

model_id  = NCAIR_MODELS[language]
lang_code = LANG_CODES[language]

print('\n' + '=' * 55)
print('  NIGERIAN MEDIA SUBTITLING PIPELINE')
print('=' * 55)
print(f'  Source    : {language} ({lang_code})')
print(f'  Target    : {target_language}')
print(f'  ASR model : {model_id}')
print('=' * 55)

# PART C — Stage 1: Transcribe

AUDIO_PATH = 'extracted_audio.wav'
OUTPUT_SRT = f'{base_name}.srt'

print('\n[Stage 1 — 1/4] Extracting full audio...')
subprocess.run([
    'ffmpeg', '-y', '-i', VIDEO_PATH,
    '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
    AUDIO_PATH, '-loglevel', 'warning',
], check=True)
print(f'    Audio ready ({os.path.getsize(AUDIO_PATH) / (1024 * 1024):.1f} MB)')

print('\n[Stage 1 — 2/4] Loading ASR pipeline...')
device = 0 if torch.cuda.is_available() else -1
pipe = hf_pipeline(
    'automatic-speech-recognition',
    model=model_id,
    device=device,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    generate_kwargs={
        'task': 'transcribe', 'language': lang_code,
        'no_repeat_ngram_size': 3, 'repetition_penalty': 1.1,
    },
)
print('  Pipeline ready.')

print('\n[Stage 1 — 3/4] Transcribing...')
audio, sr = librosa.load(AUDIO_PATH, sr=16000, mono=True)
total_duration = len(audio) / sr

CHUNK_DURATION = 30
OVERLAP        = 2
STEP           = CHUNK_DURATION - OVERLAP
chunk_starts   = np.arange(0, total_duration, STEP)
num_chunks     = len(chunk_starts)
print(f'  Audio: {total_duration:.1f}s -> {num_chunks} chunks (2s overlap)')

chunk_word_lists = []
for i, chunk_start_sec in enumerate(chunk_starts):
    chunk_end_sec = min(chunk_start_sec + CHUNK_DURATION, total_duration)
    start_sample  = int(chunk_start_sec * sr)
    end_sample    = int(chunk_end_sec * sr)
    chunk_audio   = audio[start_sample:end_sample]

    result = pipe({'raw': chunk_audio, 'sampling_rate': sr}, return_timestamps='word')

    chunk_words = []
    for word in result.get('chunks', []):
        local_start, local_end = word['timestamp']
        global_start = (local_start or 0) + chunk_start_sec
        global_end   = (local_end or global_start + 0.3) + chunk_start_sec
        chunk_words.append({
            'text': word['text'].strip(),
            'start': round(global_start, 3),
            'end': round(global_end, 3),
        })
    chunk_word_lists.append(chunk_words)
    print(f'  Chunk {i + 1}/{num_chunks} — {len(chunk_words)} words.')

del pipe; gc.collect(); torch.cuda.empty_cache()

def merge_word_lists(chunk_word_lists):
    all_words = []
    for i, words in enumerate(chunk_word_lists):
        if i == 0:
            all_words.extend(words)
            continue
        N = 8
        tail = [w['text'].lower() for w in all_words[-N:]]
        head = [w['text'].lower() for w in words[:N]]
        overlap_len = 0
        for length in range(min(len(tail), len(head)), 0, -1):
            if tail[-length:] == head[:length]:
                overlap_len = length
                break
        all_words.extend(words[overlap_len:])
    return all_words

all_words = merge_word_lists(chunk_word_lists)
print(f'  {len(all_words)} words after deduplication.')

print('\n[Stage 1 — 4/4] Grouping into subtitle blocks...')

def normalize_punctuation(text):
    return re.sub(r'\s+([.!?,;:])', r'\1', text)

PUNCTUATION_BREAK = {'.', '?', '!', ';'}
subtitles, block_words, block_start, block_end, prev_end = [], [], None, None, None

for word in all_words:
    text, start, end = word['text'], word['start'], word['end']
    if not text:
        continue
    pause = (start - prev_end) if prev_end is not None else 0
    prev_end = end
    if block_start is None:
        block_start = start
    block_words.append(text)
    block_end = end

    duration, word_count = block_end - block_start, len(block_words)
    ends_sent = any(text.endswith(p) for p in PUNCTUATION_BREAK)
    should_break = (
        duration >= max_subtitle_duration or
        pause >= pause_threshold or
        word_count >= max_words_per_subtitle or
        (ends_sent and word_count >= 3)
    )
    if should_break:
        subtitles.append({'text': normalize_punctuation(' '.join(block_words)),
                           'timestamp': (block_start, block_end)})
        block_words, block_start, block_end = [], None, None

if block_words:
    subtitles.append({'text': normalize_punctuation(' '.join(block_words)),
                       'timestamp': (block_start, block_end)})

chunks = subtitles
print('\n--- Transcript ---')
for c in chunks:
    print(c['text'])
print('-' * 55)

def secs_to_srt(s):
    if s is None: s = 0.0
    ms = int(float(s) * 1000)
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s2, ms = divmod(ms, 1_000)
    return pysrt.SubRipTime(h, m, s2, ms)

subs = pysrt.SubRipFile()
for i, c in enumerate(chunks, 1):
    txt = c.get('text', '').strip()
    if not txt:
        continue
    ts = c.get('timestamp', (0.0, 0.0))
    start, end = ts[0], ts[1]
    if end is None or end <= start:
        end = start + 3.0
    subs.append(pysrt.SubRipItem(i, secs_to_srt(start), secs_to_srt(end), txt))
subs.save(OUTPUT_SRT, encoding='utf-8')
print(f'\n Stage 1 SRT saved — {len(subs)} entries')

# PART D — Stage 2: Cleanup (N-ATLaS GGUF)

from llama_cpp import Llama
import llama_cpp

SYSTEM_PROMPTS_CLEAN = {
    'ha': ('You are a Hausa transcript restoration engine. '
           'Fix spelling, punctuation, capitalization, and obvious ASR errors only. '
           'Do NOT translate. Do NOT summarize. Do NOT paraphrase. Do NOT rewrite. '
           'Do NOT add or remove any information. '
           'If you are less than 95% confident a correction is right, leave the word unchanged. '
           'Return output in exactly this format for every entry:\n[ID]\n<restored Hausa text>\n\n'
           'Do not merge, split, delete, add, or renumber entries. '
           'No commentary. No explanations. Output only the numbered entries.'),
    'yo': ('You are a Yoruba transcript restoration engine. '
           'Fix spelling, punctuation, capitalization, and obvious ASR errors only. '
           'Do NOT translate. Do NOT summarize. Do NOT paraphrase. Do NOT rewrite. '
           'Do NOT add or remove any information. '
           'If you are less than 95% confident a correction is right, leave the word unchanged. '
           'Return output in exactly this format for every entry:\n[ID]\n<restored Yoruba text>\n\n'
           'Do not merge, split, delete, add, or renumber entries. '
           'No commentary. No explanations. Output only the numbered entries.'),
    'ig': ('You are an Igbo transcript restoration engine. '
           'Fix spelling, punctuation, capitalization, and obvious ASR errors only. '
           'Do NOT translate. Do NOT summarize. Do NOT paraphrase. Do NOT rewrite. '
           'Do NOT add or remove any information. '
           'If you are less than 95% confident a correction is right, leave the word unchanged. '
           'Return output in exactly this format for every entry:\n[ID]\n<restored Igbo text>\n\n'
           'Do not merge, split, delete, add, or renumber entries. '
           'No commentary. No explanations. Output only the numbered entries.'),
    'en': ('You are a Nigerian English transcript restoration engine. '
           'Fix spelling, punctuation, capitalization, and obvious ASR errors only. '
           'Do NOT summarize. Do NOT paraphrase. Do NOT rewrite. '
           'Do NOT add or remove any information. '
           'If you are less than 95% confident a correction is right, leave the word unchanged. '
           'Return output in exactly this format for every entry:\n[ID]\n<restored English text>\n\n'
           'Do not merge, split, delete, add, or renumber entries. '
           'No commentary. No explanations. Output only the numbered entries.'),
}

if not use_cleanup:
    print('\nSkipping Stage 2 cleanup.')
else:
    system_prompt_clean = SYSTEM_PROMPTS_CLEAN[lang_code]
    print('\n[Stage 2] Loading N-ATLaS GGUF (Q4_K_M)...')
    llm = Llama.from_pretrained(repo_id='tosinamuda/N-ATLaS-GGUF', filename='*Q4_K_M*',
                                 n_gpu_layers=-1, n_ctx=8192, verbose=False)
    print('   CUDA offload active.' if llama_cpp.llama_supports_gpu_offload()
          else '    CUDA offload NOT active.')

    def format_batch(sub_batch):
        return '\n\n'.join(f'[{local_idx}]\n{text}'
                            for local_idx, (_, text) in enumerate(sub_batch, 1))

    def run_llm_clean(batch_text):
        response = llm.create_chat_completion(
            messages=[{'role': 'system', 'content': system_prompt_clean},
                      {'role': 'user', 'content': batch_text}],
            max_tokens=1500, repeat_penalty=1.12, temperature=0.1,
        )
        return response['choices'][0]['message']['content'].strip()

    def parse_response(response_text, sub_batch):
        pattern = r'\[(\d+)\]\s*\n(.*?)(?=\n\[\d+\]|\Z)'
        matches = re.findall(pattern, response_text, re.DOTALL)
        index_map = {local: real for local, (real, _) in enumerate(sub_batch, 1)}
        parsed, seen = {}, set()
        for local_str, text in matches:
            local = int(local_str)
            if local in seen:
                return None, f'Duplicate local ID: {local}'
            if local not in index_map:
                return None, f'Unexpected local ID: {local}'
            seen.add(local)
            parsed[index_map[local]] = text.strip()
        return parsed, None

    def validate(parsed, sub_batch):
        return parsed is not None and set(parsed.keys()) == {idx for idx, _ in sub_batch}

    originals   = {s.index: s.text for s in subs}
    sub_entries = [(s.index, s.text) for s in subs]
    total_subs  = len(sub_entries)
    total_batches = (total_subs + batch_size - 1) // batch_size
    cleaned_map = {}

    print(f'\nCleaning {total_subs} subtitles in batches of {batch_size}...')
    for batch_start in range(0, total_subs, batch_size):
        sub_batch = sub_entries[batch_start: batch_start + batch_size]
        batch_num = batch_start // batch_size + 1
        success = False
        for attempt in range(max_retries + 1):
            batch_text  = format_batch(sub_batch)
            response    = run_llm_clean(batch_text)
            parsed, err = parse_response(response, sub_batch)
            if validate(parsed, sub_batch):
                cleaned_map.update(parsed)
                success = True
                break
            else:
                missing = {idx for idx, _ in sub_batch} - set((parsed or {}).keys())
                print(f'    Batch {batch_num} attempt {attempt + 1} failed '
                      f'(err={err}, missing={missing}). Retrying...')
        if not success:
            print(f'  ❌ Batch {batch_num} failed — keeping original ASR text.')
            for idx, text in sub_batch:
                cleaned_map[idx] = text
        print(f'  Batch {batch_num}/{total_batches} done.')

    del llm; gc.collect()
    print('  Model freed.')

    for s in subs:
        if s.index in cleaned_map:
            s.text = cleaned_map[s.index]

    CLEANED_SRT = f'{base_name}_cleaned.srt'
    subs.save(CLEANED_SRT, encoding='utf-8')
    print(f'\n Stage 2 cleaned SRT — {len(subs)} entries saved.')

# PART E — Stage 3: Translation (N-ATLaS GGUF)

LANG_DISPLAY_NAMES = {'ha': 'Hausa', 'yo': 'Yoruba', 'ig': 'Igbo', 'en': 'Nigerian English'}

def build_translate_system_prompt(source_lang_code, target_language_name):
    # Built dynamically (rather than a fixed lookup table) so every source/target
    # combination is supported, e.g. Hausa -> Yoruba, not just X -> English or
    # English -> X. Previously, a non-English source ignored the chosen target
    # and silently always translated to English.
    source_name = LANG_DISPLAY_NAMES[source_lang_code]
    return (
        f'You are a {source_name}-to-{target_language_name} translator. '
        f"Translate each numbered {source_name} subtitle entry into natural, accurate "
        f"{target_language_name}. "
        "Preserve the speaker's exact meaning. "
        'Do NOT summarize, paraphrase, add, or remove information. '
        'Do NOT include commentary, explanations, or notes of any kind. '
        f'Return output in exactly this format for every entry:\n[ID]\n<{target_language_name} translation>\n\n'
        'Do not merge, split, delete, add, or renumber entries. '
        'Output only the numbered entries, nothing else.'
    )

if not use_translation:
    print('\nSkipping Stage 3 translation.')
elif language == target_language:
    print(f'\nSource and target are both {target_language} — skipping translation.')
else:
    system_prompt_translate = build_translate_system_prompt(lang_code, target_language)
    if True:
        print('\n[Stage 3] Loading N-ATLaS GGUF (Q4_K_M) for translation...')
        llm = Llama.from_pretrained(repo_id='tosinamuda/N-ATLaS-GGUF', filename='*Q4_K_M*',
                                     n_gpu_layers=-1, n_ctx=8192, verbose=False)
        print('   CUDA offload active.' if llama_cpp.llama_supports_gpu_offload()
              else '    CUDA offload NOT active.')

        def format_batch_t(sub_batch):
            return '\n\n'.join(f'[{local_idx}]\n{text}'
                                for local_idx, (_, text) in enumerate(sub_batch, 1))

        def run_llm_translate(batch_text):
            response = llm.create_chat_completion(
                messages=[{'role': 'system', 'content': system_prompt_translate},
                          {'role': 'user', 'content': batch_text}],
                max_tokens=2000, repeat_penalty=1.12, temperature=0.1,
            )
            return response['choices'][0]['message']['content'].strip()

        def parse_response_t(response_text, sub_batch):
            pattern = r'\[(\d+)\]\s*\n(.*?)(?=\n\[\d+\]|\Z)'
            matches = re.findall(pattern, response_text, re.DOTALL)
            index_map = {local: real for local, (real, _) in enumerate(sub_batch, 1)}
            parsed, seen = {}, set()
            for local_str, text in matches:
                local = int(local_str)
                if local in seen:
                    return None, f'Duplicate local ID: {local}'
                if local not in index_map:
                    return None, f'Unexpected local ID: {local}'
                seen.add(local)
                parsed[index_map[local]] = text.strip()
            return parsed, None

        def validate_t(parsed, sub_batch):
            return parsed is not None and set(parsed.keys()) == {idx for idx, _ in sub_batch}

        originals_t   = {s.index: s.text for s in subs}
        sub_entries_t = [(s.index, s.text) for s in subs]
        total_subs_t  = len(sub_entries_t)
        total_batches_t = (total_subs_t + batch_size_t - 1) // batch_size_t
        translated_map = {}

        print(f'\nTranslating {total_subs_t} subtitles in batches of {batch_size_t}...')
        for batch_start in range(0, total_subs_t, batch_size_t):
            sub_batch    = sub_entries_t[batch_start: batch_start + batch_size_t]
            expected_ids = set(idx for idx, _ in sub_batch)
            batch_num    = batch_start // batch_size_t + 1
            success = False
            for attempt in range(max_retries_t + 1):
                batch_text  = format_batch_t(sub_batch)
                response    = run_llm_translate(batch_text)
                parsed, err = parse_response_t(response, sub_batch)
                if validate_t(parsed, sub_batch):
                    translated_map.update(parsed)
                    success = True
                    break
                else:
                    missing = expected_ids - set((parsed or {}).keys())
                    print(f'    Batch {batch_num} attempt {attempt + 1} failed '
                          f'(err={err}, missing={missing}). Retrying...')
            if not success:
                print(f'  ❌ Batch {batch_num} failed — keeping Stage 2 cleaned text.')
                for idx, text in sub_batch:
                    translated_map[idx] = text
            print(f'  Batch {batch_num}/{total_batches_t} done.')

        del llm; gc.collect()
        print('  Model freed.')

        for s in subs:
            if s.index in translated_map:
                original_cleaned = originals_t.get(s.index, '')
                s.text = f'{original_cleaned}\n{translated_map[s.index]}'

        TRANSLATED_SRT = f'{base_name}_translated.srt'
        subs.save(TRANSLATED_SRT, encoding='utf-8')
        print(f'\n Stage 3 translated SRT — {len(subs)} entries saved.')

# PART F — Burn captions into the video (hardsub)

if 'TRANSLATED_SRT' in globals() and os.path.exists(TRANSLATED_SRT):
    FINAL_SRT = TRANSLATED_SRT
    print(f'\n[Burn-in] Using translated SRT: {FINAL_SRT}')
elif 'CLEANED_SRT' in globals() and os.path.exists(CLEANED_SRT):
    FINAL_SRT = CLEANED_SRT
    print(f'\n[Burn-in] Using cleaned SRT: {FINAL_SRT}')
else:
    FINAL_SRT = OUTPUT_SRT
    print(f'\n[Burn-in] Using raw (uncleaned) SRT: {FINAL_SRT}')

BURNED_VIDEO = f'{base_name}_captioned.mp4'

def _escape_for_ffmpeg_filter(path):
    return path.replace('\\', '\\\\').replace(':', '\\:').replace("'", "\\'")

escaped_srt = _escape_for_ffmpeg_filter(os.path.abspath(FINAL_SRT))

probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=width,height', '-of', 'json', VIDEO_PATH],
    capture_output=True, text=True,
)
video_info = _json_probe.loads(probe.stdout)['streams'][0]
video_w, video_h = video_info['width'], video_info['height']
font_size = max(10, round(video_h * subtitle_font_percent / 100))
print(f'Video resolution: {video_w}x{video_h} -> font size {font_size}px')

subtitle_style = (
    f"FontName=Arial,FontSize={font_size},PrimaryColour=&H00FFFFFF,"
    "OutlineColour=&H00000000,BorderStyle=3,Outline=1,Shadow=0,"
    f"BackColour=&H80000000,Alignment=2,MarginV=30,"
    f"PlayResX={video_w},PlayResY={video_h}"
)

cmd = [
    'ffmpeg', '-y', '-i', VIDEO_PATH,
    '-vf', f"subtitles='{escaped_srt}':force_style='{subtitle_style}'",
    '-c:v', 'libx264', '-crf', '18', '-preset', 'medium',
    '-c:a', 'aac', '-b:a', '192k',
    BURNED_VIDEO, '-loglevel', 'error',
]

print('Burning captions into the video — this can take a while for longer videos...')
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print('❌ ffmpeg failed:')
    print(result.stderr[-3000:])
    raise RuntimeError('Burning captions failed — see the ffmpeg output above.')

print(f'\n Captioned video saved: {BURNED_VIDEO}')
print(f'   Size: {os.path.getsize(BURNED_VIDEO) / (1024 * 1024):.1f} MB')

from google.colab import files
files.download(BURNED_VIDEO)
print(' Downloading — one video file, captions burned in, nothing else needed.')


Select your video file...


Saving my_hausa_clip.mp4 to my_hausa_clip (1).mp4
✅ Video ready: my_hausa_clip (1).mp4
   Size: 9.6 MB

[Language ID] Extracting a short audio sample for detection...
[Language ID] Loading a small multilingual model just for detection...


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

[Language ID] Whisper detected code "sw" -> not one of our 4 supported languages (could be a misdetected Igbo clip, or genuine noise/silence)

  Could not confidently auto-detect the language.
Note: Igbo can't be auto-detected (see above) — double check if it's spoken.

Press Enter to accept "Nigerian English", or type the correct language (Hausa, Yoruba, Igbo, Nigerian English): Hausa
✅ Source language: Hausa

Which language should the translation be in? (English, Hausa, Yoruba, Igbo): English
✅ Target language: English

  NIGERIAN MEDIA SUBTITLING PIPELINE
  Source    : Hausa (ha)
  Target    : English
  ASR model : NCAIR1/Hausa-ASR

[Stage 1 — 1/4] Extracting full audio...
    Audio ready (2.6 MB)

[Stage 1 — 2/4] Loading ASR pipeline...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


  ✅ CUDA offload active.

Cleaning 26 subtitles in batches of 20...
  Batch 1/2 done.
  Batch 2/2 done.
  Model freed.

✅ Stage 2 cleaned SRT — 26 entries saved.

[Stage 3] Loading N-ATLaS GGUF (Q4_K_M) for translation...
  ✅ CUDA offload active.

Translating 26 subtitles in batches of 20...
  ⚠️  Batch 1 attempt 1 failed (err=None, missing={1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20}). Retrying...
  ⚠️  Batch 1 attempt 2 failed (err=None, missing={1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20}). Retrying...
  ⚠️  Batch 1 attempt 3 failed (err=None, missing={1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20}). Retrying...
  ❌ Batch 1 failed — keeping Stage 2 cleaned text.
  Batch 1/2 done.
  Batch 2/2 done.
  Model freed.

✅ Stage 3 translated SRT — 26 entries saved.

[Burn-in] Using translated SRT: my_hausa_clip (1)_translated.srt
Video resolution: 720x1280 -> font size 54px
Burning captions into the video — this c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloading — one video file, captions burned in, nothing else needed.


## Step 5: Real-Time Web App (browser UI, no Streamlit)

A small **FastAPI** backend + a single **vanilla HTML/CSS/JS** page (no React, no
Streamlit, no build step). This runs right here in Colab and is exposed publicly
through an **ngrok** tunnel — the simplest way to get a real URL out of a Colab GPU
without renting a server.

**How the "real-time" part actually works:** the moment you pick a file, the browser
plays it immediately from local memory — it doesn't wait for the server at all. In
parallel, the file is uploaded and the backend chops the audio into short chunks,
runs ASR → cleanup → translation on each chunk in turn, and pushes each finished
caption to the browser over a **WebSocket** as soon as it's ready. The video's caption
overlay only ever shows a caption once its chunk has actually finished processing —
that's the "availability ruler" under the video: it fills in live, chunk by chunk,
so you can see exactly how far captioning has caught up to playback.

This reuses the same ASR + N-ATLaS loading pattern as Step 6, just applied to an
uploaded file's timeline instead of a live microphone.

**Before running:** get a free ngrok authtoken at ngrok.com → add it to Colab's
Secrets manager as `NGROK_AUTHTOKEN` (same place as `HF_TOKEN`).

In [ ]:
#@markdown ### Install web app dependencies + write the frontend file
!pip install -q fastapi "uvicorn[standard]" python-multipart pyngrok nest_asyncio

import os
os.makedirs('/content/webapp/static', exist_ok=True)

FRONTEND_HTML = "<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"UTF-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">\n<title>N-ATLaS Caption Desk</title>\n<link rel=\"preconnect\" href=\"https://fonts.googleapis.com\">\n<link href=\"https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500;600&family=Inter:wght@400;500;600;700&display=swap\" rel=\"stylesheet\">\n<style>\n  :root{\n    --bg:#0F1216;\n    --panel:#171C22;\n    --panel-2:#1D242C;\n    --line:#2A323C;\n    --text:#E8ECEF;\n    --text-dim:#8A93A0;\n    --teal:#2DD9C4;\n    --amber:#FF9640;\n    --red:#FF5C5C;\n    --radius:10px;\n  }\n  *{box-sizing:border-box;}\n  html,body{margin:0;padding:0;}\n  body{\n    background:\n      radial-gradient(1200px 600px at 15% -10%, #151B21 0%, transparent 60%),\n      var(--bg);\n    color:var(--text);\n    font-family:'Inter',system-ui,sans-serif;\n    min-height:100vh;\n    padding:32px 20px 60px;\n  }\n  .wrap{max-width:1080px;margin:0 auto;}\n\n  header{display:flex;align-items:baseline;justify-content:space-between;flex-wrap:wrap;gap:8px;margin-bottom:28px;}\n  .brand{display:flex;align-items:center;gap:10px;}\n  .brand-dot{width:9px;height:9px;border-radius:50%;background:var(--teal);box-shadow:0 0 10px var(--teal);}\n  h1{font-size:19px;font-weight:600;letter-spacing:0.01em;margin:0;}\n  .sub{color:var(--text-dim);font-size:13px;margin-top:2px;}\n  .tag{font-family:'IBM Plex Mono',monospace;font-size:11px;color:var(--text-dim);border:1px solid var(--line);border-radius:20px;padding:5px 12px;letter-spacing:0.04em;}\n\n  .panel{\n    background:linear-gradient(180deg,var(--panel),var(--panel-2));\n    border:1px solid var(--line);\n    border-radius:var(--radius);\n    padding:20px;\n  }\n\n  .setup{display:grid;grid-template-columns:1.4fr 1fr 1fr auto;gap:14px;align-items:end;margin-bottom:18px;}\n  @media (max-width:760px){.setup{grid-template-columns:1fr 1fr;}}\n  .field label{display:block;font-size:11px;text-transform:uppercase;letter-spacing:0.06em;color:var(--text-dim);margin-bottom:6px;}\n  .field select, .field input[type=file]{\n    width:100%;background:#11151A;border:1px solid var(--line);color:var(--text);\n    border-radius:7px;padding:9px 10px;font-family:'Inter',sans-serif;font-size:13px;\n  }\n  .field.check{display:flex;align-items:center;gap:8px;padding-bottom:8px;}\n  .field.check input{accent-color:var(--teal);width:15px;height:15px;}\n  .field.check label{margin:0;font-size:12.5px;color:var(--text);text-transform:none;letter-spacing:0;}\n\n  button{\n    background:var(--teal);color:#06201C;border:none;border-radius:7px;\n    font-weight:600;font-size:13.5px;padding:10px 18px;cursor:pointer;\n    transition:filter .15s ease, transform .1s ease;\n  }\n  button:hover{filter:brightness(1.08);}\n  button:active{transform:scale(0.98);}\n  button:disabled{background:var(--line);color:var(--text-dim);cursor:not-allowed;}\n\n  .stage{display:none;margin-top:22px;}\n  .stage.active{display:block;}\n\n  .status-row{display:flex;align-items:center;gap:10px;margin-bottom:12px;font-family:'IBM Plex Mono',monospace;font-size:12px;color:var(--text-dim);}\n  .status-dot{width:8px;height:8px;border-radius:50%;background:var(--amber);animation:pulse 1.3s infinite ease-in-out;}\n  .status-dot.done{background:var(--teal);animation:none;}\n  @keyframes pulse{0%,100%{opacity:1;}50%{opacity:.35;}}\n\n  .stage-grid{display:grid;grid-template-columns:1fr 300px;gap:18px;}\n  @media (max-width:820px){.stage-grid{grid-template-columns:1fr;}}\n\n  .video-frame{position:relative;border-radius:var(--radius);overflow:hidden;background:#000;border:1px solid var(--line);}\n  video{display:block;width:100%;max-height:480px;background:#000;}\n  .cap-overlay{\n    position:absolute;left:6%;right:6%;bottom:8%;text-align:center;pointer-events:none;\n    display:flex;flex-direction:column;gap:4px;\n  }\n  .cap-line{\n    display:inline-block;align-self:center;\n    background:rgba(6,10,14,0.72);backdrop-filter:blur(2px);\n    color:#fff;font-weight:600;font-size:16px;line-height:1.3;\n    padding:5px 12px;border-radius:5px;text-shadow:0 1px 3px rgba(0,0,0,.6);\n  }\n  .cap-line.translated{font-weight:500;color:var(--teal);font-size:14px;}\n\n  .ruler-label{font-family:'IBM Plex Mono',monospace;font-size:10.5px;color:var(--text-dim);\n    display:flex;justify-content:space-between;margin:10px 2px 4px;letter-spacing:0.03em;}\n  .ruler{position:relative;height:9px;border-radius:5px;background:#11151A;border:1px solid var(--line);overflow:hidden;}\n  .ruler-fill{position:absolute;top:0;left:0;height:100%;background:linear-gradient(90deg,var(--teal),#54F0DB);width:0%;transition:width .3s ease;}\n  .ruler-playhead{position:absolute;top:-3px;width:2px;height:15px;background:var(--amber);left:0%;box-shadow:0 0 6px var(--amber);}\n\n  .log{\n    background:#11151A;border:1px solid var(--line);border-radius:var(--radius);\n    height:480px;overflow-y:auto;padding:14px;font-family:'IBM Plex Mono',monospace;font-size:12px;\n  }\n  .log-title{font-family:'Inter',sans-serif;font-size:11px;text-transform:uppercase;letter-spacing:0.06em;\n    color:var(--text-dim);margin-bottom:10px;}\n  .log-item{padding:9px 0;border-bottom:1px solid var(--line);}\n  .log-item:last-child{border-bottom:none;}\n  .log-time{color:var(--amber);}\n  .log-native{color:var(--text);margin-top:3px;}\n  .log-translated{color:var(--teal);margin-top:2px;}\n  .log-empty{color:var(--text-dim);font-style:italic;}\n\n  .error{color:var(--red);font-size:12.5px;margin-top:10px;font-family:'IBM Plex Mono',monospace;}\n</style>\n</head>\n<body>\n<div class=\"wrap\">\n  <header>\n    <div class=\"brand\">\n      <span class=\"brand-dot\"></span>\n      <div>\n        <h1>N-ATLaS Caption Desk</h1>\n        <div class=\"sub\">Nigerian-language ASR &amp; translation, captions filling in as they're generated</div>\n      </div>\n    </div>\n    <span class=\"tag\">HAUSA \u00b7 YORUBA \u00b7 IGBO \u00b7 NG-ENGLISH</span>\n  </header>\n\n  <div class=\"panel\">\n    <div class=\"setup\">\n      <div class=\"field\">\n        <label for=\"fileInput\">Video file</label>\n        <input type=\"file\" id=\"fileInput\" accept=\"video/*\">\n      </div>\n      <div class=\"field\">\n        <label for=\"sourceLang\">Spoken language</label>\n        <select id=\"sourceLang\">\n          <option>Hausa</option>\n          <option>Yoruba</option>\n          <option>Igbo</option>\n          <option selected>Nigerian English</option>\n        </select>\n      </div>\n      <div class=\"field\">\n        <label for=\"targetLang\">Translate to</label>\n        <select id=\"targetLang\">\n          <option selected>English</option>\n          <option>Hausa</option>\n          <option>Yoruba</option>\n          <option>Igbo</option>\n        </select>\n      </div>\n      <button id=\"startBtn\" disabled>\u25b6 Start captioning</button>\n    </div>\n    <div class=\"field check\">\n      <input type=\"checkbox\" id=\"translateToggle\" checked>\n      <label for=\"translateToggle\">Show translated line under native caption</label>\n    </div>\n    <div id=\"errorBox\" class=\"error\"></div>\n  </div>\n\n  <div class=\"stage\" id=\"stage\">\n    <div class=\"status-row\">\n      <span class=\"status-dot\" id=\"statusDot\"></span>\n      <span id=\"statusText\">CONNECTING\u2026</span>\n    </div>\n\n    <div class=\"stage-grid\">\n      <div>\n        <div class=\"video-frame\">\n          <video id=\"video\" controls playsinline></video>\n          <div class=\"cap-overlay\" id=\"capOverlay\"></div>\n        </div>\n        <div class=\"ruler-label\"><span>CAPTION AVAILABILITY</span><span id=\"rulerPct\">0%</span></div>\n        <div class=\"ruler\">\n          <div class=\"ruler-fill\" id=\"rulerFill\"></div>\n          <div class=\"ruler-playhead\" id=\"rulerPlayhead\"></div>\n        </div>\n      </div>\n\n      <div class=\"log\">\n        <div class=\"log-title\">Live transcript log</div>\n        <div id=\"logList\"><div class=\"log-empty\">Waiting for the first caption\u2026</div></div>\n      </div>\n    </div>\n  </div>\n</div>\n\n<script>\nconst fileInput = document.getElementById('fileInput');\nconst startBtn = document.getElementById('startBtn');\nconst stage = document.getElementById('stage');\nconst video = document.getElementById('video');\nconst capOverlay = document.getElementById('capOverlay');\nconst statusDot = document.getElementById('statusDot');\nconst statusText = document.getElementById('statusText');\nconst rulerFill = document.getElementById('rulerFill');\nconst rulerPct = document.getElementById('rulerPct');\nconst rulerPlayhead = document.getElementById('rulerPlayhead');\nconst logList = document.getElementById('logList');\nconst errorBox = document.getElementById('errorBox');\nconst sourceLang = document.getElementById('sourceLang');\nconst targetLang = document.getElementById('targetLang');\nconst translateToggle = document.getElementById('translateToggle');\n\nlet captions = [];   // {start, end, native, translated}\nlet videoDuration = 0;\nlet done = false;\nlet bufferedUntil = 0;      // furthest time we have a caption for\nlet hasStartedPlayback = false;\nlet autoPaused = false;     // paused by us (buffering), not by the user\n\nfileInput.addEventListener('change', () => {\n  startBtn.disabled = !fileInput.files.length;\n});\n\nfunction fmtTime(s){\n  s = Math.max(0, s || 0);\n  const m = Math.floor(s/60), sec = Math.floor(s%60);\n  return String(m).padStart(2,'0') + ':' + String(sec).padStart(2,'0');\n}\n\nfunction addLogEntry(cap){\n  if (logList.querySelector('.log-empty')) logList.innerHTML = '';\n  const div = document.createElement('div');\n  div.className = 'log-item';\n  const translatedHtml = cap.translated ? `<div class=\"log-translated\">${cap.translated}</div>` : '';\n  div.innerHTML = `<span class=\"log-time\">${fmtTime(cap.start)}</span>\n    <div class=\"log-native\">${cap.native || '(silence)'}</div>${translatedHtml}`;\n  logList.appendChild(div);\n  logList.scrollTop = logList.scrollHeight;\n}\n\nfunction updateRuler(){\n  if (!videoDuration) return;\n  const lastEnd = captions.length ? captions[captions.length-1].end : 0;\n  const pct = Math.min(100, (lastEnd / videoDuration) * 100);\n  rulerFill.style.width = pct.toFixed(1) + '%';\n  rulerPct.textContent = Math.round(pct) + '%';\n}\n\nvideo.addEventListener('timeupdate', () => {\n  if (videoDuration) {\n    rulerPlayhead.style.left = Math.min(100, (video.currentTime/videoDuration)*100) + '%';\n  }\n  const t = video.currentTime;\n  const active = captions.find(c => t >= c.start && t < c.end);\n  if (active) {\n    let html = `<span class=\"cap-line\">${active.native || ''}</span>`;\n    if (translateToggle.checked && active.translated) {\n      html += `<span class=\"cap-line translated\">${active.translated}</span>`;\n    }\n    capOverlay.innerHTML = html;\n  } else {\n    capOverlay.innerHTML = '';\n  }\n  if (!done && hasStartedPlayback && !video.paused && t >= bufferedUntil - 0.15) {\n    video.pause();\n    autoPaused = true;\n    statusText.textContent = 'BUFFERING \u2014 waiting for captions to catch up\u2026';\n  }\n});\n\nvideo.addEventListener('loadedmetadata', () => {\n  videoDuration = video.duration;\n  updateRuler();\n});\n\nstartBtn.addEventListener('click', async () => {\n  const file = fileInput.files[0];\n  if (!file) return;\n  errorBox.textContent = '';\n  startBtn.disabled = true;\n  startBtn.textContent = 'Uploading\u2026';\n\n  video.src = URL.createObjectURL(file);\n  stage.classList.add('active');\n  statusText.textContent = 'PROCESSING \u2014 waiting for first caption\u2026';\n\n  const form = new FormData();\n  form.append('file', file);\n  form.append('source_lang', sourceLang.value);\n  form.append('target_lang', targetLang.value);\n  form.append('translate', translateToggle.checked);\n\n  try {\n    const res = await fetch('/upload', { method: 'POST', body: form });\n    if (!res.ok) throw new Error('Upload failed: ' + res.status);\n    const { session_id } = await res.json();\n    connectSocket(session_id);\n  } catch (err) {\n    errorBox.textContent = '\u26a0\ufe0f ' + err.message;\n    statusText.textContent = 'UPLOAD FAILED';\n  }\n});\n\nfunction connectSocket(sessionId){\n  const proto = location.protocol === 'https:' ? 'wss' : 'ws';\n  const ws = new WebSocket(`${proto}://${location.host}/ws/${sessionId}`);\n\n  ws.onopen = () => { statusText.textContent = 'PROCESSING \u2014 captions filling in\u2026'; };\n\n  ws.onmessage = (evt) => {\n    const msg = JSON.parse(evt.data);\n    if (msg.done) {\n      done = true;\n      statusDot.classList.add('done');\n      statusText.textContent = 'DONE \u2014 full transcript ready';\n      return;\n    }\n    captions.push(msg);\n    bufferedUntil = Math.max(bufferedUntil, msg.end);\n    addLogEntry(msg);\n    updateRuler();\n\n    if (!hasStartedPlayback) {\n      hasStartedPlayback = true;\n      statusText.textContent = 'PROCESSING \u2014 captions filling in\u2026';\n      video.play().catch(()=>{});\n    } else if (autoPaused && video.currentTime < bufferedUntil - 0.15) {\n      autoPaused = false;\n      statusText.textContent = 'PROCESSING \u2014 captions filling in\u2026';\n      video.play().catch(()=>{});\n    }\n  };\n\n  ws.onerror = () => { errorBox.textContent = '\u26a0\ufe0f Connection to the caption stream dropped.'; };\n  ws.onclose = () => { if (!done) statusText.textContent = 'DISCONNECTED'; };\n}\n</script>\n</body>\n</html>\n"

with open('/content/webapp/static/index.html', 'w', encoding='utf-8') as f:
    f.write(FRONTEND_HTML)

print('✅ Frontend written to /content/webapp/static/index.html')


✅ Frontend written to /content/webapp/static/index.html


In [ ]:
#@markdown ### Backend: FastAPI app (chunked ASR + N-ATLaS pipeline, streamed over WebSocket)

import asyncio, subprocess, uuid, os, re, gc, time
from concurrent.futures import ThreadPoolExecutor

import torch, librosa
from fastapi import FastAPI, UploadFile, Form, File, WebSocket, WebSocketDisconnect
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from transformers import pipeline as hf_pipeline
from llama_cpp import Llama
import llama_cpp

WEBAPP_NCAIR_MODELS = {
    'Hausa':            'NCAIR1/Hausa-ASR',
    'Yoruba':           'NCAIR1/Yoruba-ASR',
    'Igbo':             'NCAIR1/Igbo-ASR',
    'Nigerian English': 'NCAIR1/NigerianAccentedEnglish',
}
WEBAPP_LANG_CODES = {'Hausa': 'ha', 'Yoruba': 'yo', 'Igbo': 'ig', 'Nigerian English': 'en'}
WEBAPP_LANG_NAMES = {'ha': 'Hausa', 'yo': 'Yoruba', 'ig': 'Igbo', 'en': 'Nigerian-accented English'}

CHUNK_SECONDS = 6
EXECUTOR = ThreadPoolExecutor(max_workers=1)  # single GPU: process chunks one at a time

_asr_cache = {}
_llm = None

def get_asr_pipeline(language_name):
    if language_name not in _asr_cache:
        device = 0 if torch.cuda.is_available() else -1
        print(f'[backend] loading ASR model for {language_name} ...')
        _asr_cache[language_name] = hf_pipeline(
            'automatic-speech-recognition',
            model=WEBAPP_NCAIR_MODELS[language_name],
            device=device,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            generate_kwargs={
                'task': 'transcribe',
                'language': WEBAPP_LANG_CODES[language_name],
                'no_repeat_ngram_size': 3,
                'repetition_penalty': 1.1,
            },
        )
    return _asr_cache[language_name]

def get_llm():
    global _llm
    if _llm is None:
        print('[backend] loading N-ATLaS GGUF (Q4_K_M) ...')
        _llm = Llama.from_pretrained(
            repo_id='tosinamuda/N-ATLaS-GGUF', filename='*Q4_K_M*',
            n_gpu_layers=-1, n_ctx=4096, verbose=False,
        )
        print('  ✅ CUDA offload active.' if llama_cpp.llama_supports_gpu_offload()
              else '  ⚠️  CUDA offload NOT active — this will be slow.')
    return _llm

def build_system_prompt(source_lang_code, do_translate, target_language_name):
    source_name = WEBAPP_LANG_NAMES[source_lang_code]
    fidelity_rule = (
        "CRITICAL: this is a short fragment cut from continuous speech, not a full "
        "sentence. Do NOT invent words, do NOT complete it into a full sentence, and do "
        "NOT add any idea, detail, or fact that is not literally present in the input. "
        "Fix only spelling, obvious mis-heard words, punctuation, and capitalization."
    )
    if not do_translate:
        return (f"You clean up a raw {source_name} speech-to-text fragment. {fidelity_rule} "
                "Never translate. Output ONLY the corrected text, nothing else.")
    return (
        f"You process a raw {source_name} speech-to-text fragment in ONE step:\n"
        f"1) Clean it up. {fidelity_rule}\n"
        f"2) Translate the corrected text into {target_language_name}, staying equally "
        "literal — do not add or complete anything the translation implies but the "
        "original didn't say.\n"
        "Respond in EXACTLY this format and nothing else:\n"
        "CLEANED: <corrected text>\nTRANSLATED: <translation>"
    )

def process_utterance(raw_text, source_lang_code, do_translate, target_language_name, llm):
    if not raw_text.strip():
        return '', ''
    system_prompt = build_system_prompt(source_lang_code, do_translate, target_language_name)
    resp = llm.create_chat_completion(
        messages=[{'role': 'system', 'content': system_prompt},
                  {'role': 'user', 'content': raw_text}],
        max_tokens=350, repeat_penalty=1.12, temperature=0.1,
    )
    out = resp['choices'][0]['message']['content'].strip()
    if not do_translate:
        return (out if out else raw_text), ''
    m = re.search(r'CLEANED:\s*(.*?)\n\s*TRANSLATED:\s*(.*)', out, re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return (out if out else raw_text), ''

app = FastAPI()
SESSIONS = {}  # session_id -> {"captions": [...], "clients": set(), "status": "processing"|"done"}

@app.get('/')
async def root():
    return FileResponse('/content/webapp/static/index.html')

async def broadcast(session_id, message):
    dead = []
    for ws in SESSIONS[session_id]['clients']:
        try:
            await ws.send_json(message)
        except Exception:
            dead.append(ws)
    for ws in dead:
        SESSIONS[session_id]['clients'].discard(ws)

async def process_video(session_id, video_path, source_lang, target_lang, do_translate):
    session = SESSIONS[session_id]
    loop = asyncio.get_event_loop()
    try:
        audio_path = video_path + '_audio.wav'
        await loop.run_in_executor(EXECUTOR, lambda: subprocess.run(
            ['ffmpeg', '-y', '-i', video_path, '-ar', '16000', '-ac', '1',
             audio_path, '-loglevel', 'error'], check=True))

        audio, sr = librosa.load(audio_path, sr=16000, mono=True)
        total_duration = len(audio) / sr

        asr = await loop.run_in_executor(EXECUTOR, get_asr_pipeline, source_lang)
        llm = await loop.run_in_executor(EXECUTOR, get_llm)
        source_lang_code = WEBAPP_LANG_CODES[source_lang]

        t = 0.0
        while t < total_duration:
            end = min(t + CHUNK_SECONDS, total_duration)
            chunk = audio[int(t * sr):int(end * sr)]

            result = await loop.run_in_executor(
                EXECUTOR, lambda c=chunk: asr({'raw': c, 'sampling_rate': sr}))
            raw_text = (result.get('text') or '').strip()

            cleaned, translated = await loop.run_in_executor(
                EXECUTOR, process_utterance, raw_text, source_lang_code,
                do_translate, target_lang, llm)

            caption = {'start': round(t, 2), 'end': round(end, 2),
                       'native': cleaned, 'translated': translated}
            session['captions'].append(caption)
            await broadcast(session_id, caption)
            t = end

        session['status'] = 'done'
        await broadcast(session_id, {'done': True})
    except Exception as e:
        print(f'[backend] processing error for {session_id}: {e}')
        await broadcast(session_id, {'done': True, 'error': str(e)})
        session['status'] = 'error'

@app.post('/upload')
async def upload(file: UploadFile = File(...), source_lang: str = Form(...),
                  target_lang: str = Form(...), translate: str = Form(...)):
    session_id = uuid.uuid4().hex
    SESSIONS[session_id] = {'captions': [], 'clients': set(), 'status': 'processing'}

    os.makedirs('/content/webapp/uploads', exist_ok=True)
    video_path = f'/content/webapp/uploads/{session_id}_{file.filename}'
    with open(video_path, 'wb') as f:
        f.write(await file.read())

    do_translate = str(translate).lower() in ('true', '1', 'yes', 'on')
    asyncio.create_task(process_video(session_id, video_path, source_lang, target_lang, do_translate))

    return {'session_id': session_id}

@app.websocket('/ws/{session_id}')
async def ws_endpoint(websocket: WebSocket, session_id: str):
    await websocket.accept()
    if session_id not in SESSIONS:
        await websocket.close()
        return
    session = SESSIONS[session_id]
    session['clients'].add(websocket)
    for cap in session['captions']:            # replay anything already generated
        await websocket.send_json(cap)
    if session['status'] in ('done', 'error'):
        await websocket.send_json({'done': True})
    try:
        while True:
            await websocket.receive_text()      # just keeps the connection open
    except WebSocketDisconnect:
        session['clients'].discard(websocket)

print('✅ FastAPI app defined.')


✅ FastAPI app defined.


In [ ]:
#@markdown ### Launch: ngrok tunnel + uvicorn server
import nest_asyncio, uvicorn
from pyngrok import ngrok, conf

try:
    from google.colab import userdata
    NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    NGROK_AUTHTOKEN = None

if not NGROK_AUTHTOKEN:
    import getpass
    NGROK_AUTHTOKEN = getpass.getpass('Enter your ngrok authtoken (input hidden): ')

conf.get_default().auth_token = NGROK_AUTHTOKEN

ngrok.kill()  # clear any stale tunnels from a previous run
public_url = ngrok.connect(8000, "http")
print('=' * 55)
print(f'  🌍 Public URL: {public_url}')
print('=' * 55)
print('Open that link in a browser tab — that is your web app.')
print('Leave this cell running; stopping it shuts the server down.')

nest_asyncio.apply()
config = uvicorn.Config(app, host='0.0.0.0', port=8000)
server = uvicorn.Server(config)
await server.serve()


INFO:     Started server process [466]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


  🌍 Public URL: NgrokTunnel: "https://aghast-underwent-hypnotic.ngrok-free.dev" -> "http://localhost:8000"
Open that link in a browser tab — that is your web app.
Leave this cell running; stopping it shuts the server down.
INFO:     102.91.4.61:0 - "GET / HTTP/1.1" 200 OK
INFO:     102.91.4.61:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     102.91.4.61:0 - "POST /upload HTTP/1.1" 200 OK


INFO:     102.91.4.61:0 - "WebSocket /ws/74494b9526bb4a04a3a4ec3f6009f20c" [accepted]
INFO:     connection open


[backend] loading ASR model for Nigerian English ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

INFO:     102.91.4.61:0 - "GET / HTTP/1.1" 200 OK
INFO:     102.91.4.61:0 - "GET / HTTP/1.1" 200 OK
INFO:     102.91.4.61:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     102.91.4.61:0 - "POST /upload HTTP/1.1" 200 OK
INFO:     102.91.4.61:0 - "GET / HTTP/1.1" 200 OK
INFO:     102.91.4.61:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     102.91.4.61:0 - "GET / HTTP/1.1" 200 OK
INFO:     102.91.4.61:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     102.91.4.61:0 - "POST /upload HTTP/1.1" 200 OK


INFO:     102.91.4.61:0 - "WebSocket /ws/c38ff8e444ad48b98182fe3df5c5f567" [accepted]
INFO:     connection open


INFO:     102.91.4.61:0 - "GET / HTTP/1.1" 200 OK
INFO:     102.91.4.61:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     102.91.4.61:0 - "POST /upload HTTP/1.1" 200 OK


INFO:     102.91.4.61:0 - "WebSocket /ws/b35dd9c2b4114f7da4ca22dbe50d66ed" [accepted]
INFO:     connection open
